In [78]:
import pandas as pd

def carregar_bases():
    '''
    - Carrega GeoPlan, InfraGest e SisLic com colunas específicas.
    - Consolida todas as abas do AuditReport em um único DataFrame.
    - Realiza o Left Join da base Atual com a Anterior.

    Params:
        Nenhum (Lê arquivos do diretório 'bases/').

    Return:
        tuple: (df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport)
    '''
    # Lista com caminhos dos arquivos de bases
    df_geoPlan = pd.read_excel(r"bases\GeoPlan.xlsx", usecols=['ID_ESTACAO', 'TIPO_ESTACAO_GEOPLAN'])
    df_infraGest = pd.read_excel(r"bases\InfraGest.xlsx", usecols=['PONTO_ER', 'STATUS_INFRAGEST'])
    df_sisLic = pd.read_excel(r"bases\SisLic.xlsx", usecols=['CODIGO_ER', 'STATUS_SISLIC'])
    df_connectedPrevious = pd.read_excel(r"bases\t_conectados_anterior.xlsx")
    df_connectedCurrent = pd.read_excel(r"bases\t_conectados_atual.xlsx", usecols=['ID_ESTACAO'])

    # Aba 1 do arquivo AuditReport
    df_auditReport = pd.concat(
        pd.read_excel(r"bases\AuditReport.xlsx", sheet_name=None, usecols=['CODIGO_ER 1', 'TIPO_DE_PONTO 62', 'LATITUDE 25', 'LONGITUDE 27']).values(),
        ignore_index=True
    )
    
    # Faz merge do connectedCurrent com conectadosAntigo
    df_connected = df_connectedCurrent.merge(df_connectedPrevious, on='ID_ESTACAO', how='left')

    return df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport

df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport = carregar_bases()

In [79]:
def juntar_bases(df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport):
    '''
    - Realiza o merge entre a base mestre e as bases auxiliares e remove colunas de ID duplicadas (PONTO_ER, CODIGO_ER 1)
    - Aplica o prefixo 'AUX_' para diferenciar colunas auxiliares das originais.

    Params:
        df_geoPlan (pd.DataFrame): Dados de tipologia oficial do GeoPlan.
        df_infraGest (pd.DataFrame): Status de infraestrutura e conexão.
        df_sisLic (pd.DataFrame): Base de licenciamento (retornada para tratamento posterior).
        df_connected (pd.DataFrame): Base mestre de conectados (Atual + Anterior).
        df_auditReport (pd.DataFrame): Dados consolidados de auditoria física.

    Return:
        tuple: (df_connected enriquecido, df_sisLic)
    '''
    # juntar bases conectados
    df_connected = (
        df_connected
        .merge(df_geoPlan, on= 'ID_ESTACAO', how= 'left', suffixes=("", '_2'))
        .merge(df_infraGest, left_on= 'ID_ESTACAO', right_on= 'PONTO_ER', how= 'left', suffixes=("", '_2'))
        .merge(df_auditReport, left_on= 'ID_ESTACAO', right_on= 'CODIGO_ER 1', how= 'left', suffixes=("", '_2'))
        .drop(columns=['PONTO_ER', 'CODIGO_ER 1'], errors='ignore')
        )

    # Adiciona prefixo AUX_ nas colunas auxiliares
    df_connected = df_connected.rename(columns={
        'TIPO_ESTACAO_GEOPLAN_2': 'AUX_TIPO_ESTACAO_GEOPLAN', 
        'STATUS_INFRAGEST_2': 'AUX_STATUS_INFRAGEST',
        'TIPO_DE_PONTO 62_2': 'AUX_TIPO_DE_PONTO',
        'LATITUDE 25': 'AUX_LATITUDE',
        'LONGITUDE 27': 'AUX_LONGITUDE'
        })

    return df_connected, df_sisLic

df_connected, df_sisLic = juntar_bases(df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport)

In [80]:
def tratar_geoPlan(df_connected):
    '''
    '''
    # Todos os valores nulos em ['AUX_TIPO_ESTACAO_GEOPLAN'] recebem o valor A DEFINIR.
    df_connected['AUX_TIPO_ESTACAO_GEOPLAN'] = df_connected['AUX_TIPO_ESTACAO_GEOPLAN'].fillna('A DEFINIR')
    
    # Se o valor na coluna [TIPO_ESTACAO_GEOPLAN] for igual a A DEFINIR, mas não em [AUX_TIPO_ESTACAO_GEOPLAN], então [TIPO_ESTACAO_GEOPLAN] é igual a [AUX_TIPO_ESTACAO_GEOPLAN]
    FILTRO = (df_connected['TIPO_ESTACAO_GEOPLAN'] == "A DEFINIR") & (df_connected['AUX_TIPO_ESTACAO_GEOPLAN'] != "A DEFINIR")
    df_connected.loc[FILTRO, 'TIPO_ESTACAO_GEOPLAN'] = df_connected['AUX_TIPO_ESTACAO_GEOPLAN']

    return df_connected

df_connected = tratar_geoPlan(df_connected)

In [ ]:
def tratar_sisLic(df_sisLic, df_connected):
        # Prioriza os valores na coluna ['STATUS_SISLIC']
        prioridade = {
                'APROVADO': 1,
                'EM ANALISE': 2,
        }

        # Ordena as chaves pela prioridade e exclui as outras, mantendo apenas a mais importante encontrada
        df_sisLic['PRIORIDADE'] = df_sisLic['STATUS_SISLIC'].map(prioridade).fillna(99)
        df_sisLic = (
                df_sisLic
                .sort_values(by=['CODIGO_ER', 'PRIORIDADE'])
                .drop_duplicates(subset='CODIGO_ER', keep='first')
                .drop(columns=['PRIORIDADE'])
        )
 
        # 4.1 - Se o valor na coluna [STATUS_SISLIC] e [AUX_STATUS_SISLIC] forem iguais, então não faz nada
        # 4.2 - Se o valor na coluna [STATUS_SISLIC] for igual a APROVADO e [AUX_STATUS_SISLIC] não for APROVADO, então enviar para tabela de validação
        # 4.3 - Se o valor na coluna [STATUS_SISLIC] for diferente de APROVADO, então substitui o valor de [STATUS_SISLIC] pelo valor de [AUX_STATUS_SISLIC]

        # fazer merge com connected
        df_sisLic = df_sisLic.rename(columns={'STATUS_SISLIC': 'AUX_STATUS_SISLIC'})
        df_connected = df_connected.merge(df_sisLic, left_on='ID_ESTACAO', right_on='CODIGO_ER', how='left')


        return df_connected

df_connected = tratar_sisLic(df_sisLic, df_connected)


In [84]:
def tratar_auditReport(df_connected):
        df_connected = df_connected.drop(columns=['CODIGO_ER'])
        
        pass

In [89]:
def tratar_infraGest():
        pass

In [90]:
def classificar_ERs():
        pass

In [91]:
def definir_faturamento():
        pass

In [ ]:
def main():
    print("Carregando bases...")
    df_geoPlan, df_infraGest, df_sisLic, df_conectados, df_report = carregar_bases()

    print("Juntando bases...")
    df_conectados, df_sisLic = juntar_bases(df_geoPlan, df_infraGest, df_sisLic, df_conectados, df_report)

    print("Tratando base geoPlan...")
    df_connected = tratar_geoPlan(df_connected)

    # ** SisLic possui chaves repetidas em [CODIGO_ER] com status diferentes. Não devem ser excluídas de imediato
        # Antes da comparação:
        #     caso encontre alguma chave com Status = APROVADO, então exclui outras chaves duplicadas
        #     se não tiver APROVADO busca EM ANÁLISE e exclui chaves duplicadas

        # 4.1 - Se o valor na coluna [STATUS_SISLIC] e [AUX_STATUS_SISLIC] forem iguais, então não faz nada
        # 4.2 - Se o valor na coluna [STATUS_SISLIC] for igual a APROVADO e [AUX_STATUS_SISLIC] não for APROVADO, então enviar para tabela de validação
        # 4.3 - Se o valor na coluna [STATUS_SISLIC] for diferente de APROVADO, então substitui o valor de [STATUS_SISLIC] pelo valor de [AUX_STATUS_SISLIC]
    tratar_sisLic()

    # 3.1 - Se o valor na coluna [TIPO_ESTACAO_GEOPLAN] e [AUX_TIPO_DE_PONTO 62] tiverem o mesmo valor, nada é feito
    #     3.2 - Se o valor na coluna [TIPO_ESTACAO_GEOPLAN] for igual a A DEFINIR, mas não em [AUX_TIPO_DE_PONTO 62], então [TIPO_ESTACAO_GEOPLAN] é igual a [AUX_TIPO_DE_PONTO 62]
    #     3.3 - Mantém os valores de [AUX_LATITUDE 25] e [AUX_LONGITUDE 27] e o que estiver vazio nas colunas originais de coordenadas recebe os valores provindos da auditoria
    tratar_auditReport()

    # 5.1 - Se o valor na coluna [SISTEMA_ER] for igual a CONECTADO e [AUX_STATUS_INFRAGEST] for igual a CONECTADO, então nada é feito
    #     5.2 - Se o valor na coluna [SISTEMA_ER] estiver vazio, ele recebe o valor de [AUX_STATUS_INFRAGEST]
    #     5.3 - Se o valor na coluna [SISTEMA_ER] for igual a CONECTADO e [AUX_STATUS_INFRAGEST] for diferente de CONECTADO, então é enviado para validação
    #     5.4 - Se o valor na coluna [SISTEMA_ER] estiver vazio e o valor de [AUX_STATUS_INFRAGEST] também, então [SISTEMA_ER] é igual a INFRA NÃO CAPACITADA
    tratar_infraGest()

    # 6.1 - Se o valor na coluna [STATUS E-MAIL] estiver preenchido mantém a informação do [STATUS CONSOLIDADO] atual
    #     6.2 - Se o valor na coluna [AUX_TIPO_ESTACAO_GEOPLAN] for igual a SHOPPING ou SUPERMERCADO ou PONTO ECOLÓGICO, então [STATUS CONSOLIDADO] é igual a ESTAÇÕES SUSTENTÁVEIS (ENERGIA SOLAR)
    #     6.3 - Se o valor na coluna [AUX_TIPO_ESTACAO_GEOPLAN] for igual a INTERNO, então [STATUS CONSOLIDADO] é igual a CARREGADOR LENTO - INTERNO
    #     6.4 - Se o valor na coluna [AUX_TIPO_ESTACAO_GEOPLAN] for igual a COBERTURA, então [STATUS CONSOLIDADO] é igual a ESTAÇÃO EM COBERTURA
    #     6.5 - Se o valor na coluna [AUX_TIPO_ESTACAO_GEOPLAN] for igual a A DEFINIR ou NOVO_TERRENO ou COMERCIAL e [AUX_STATUS_SISLIC] for igual a APROVADO, então [STATUS CONSOLIDADO] é igual a OPERACIONAL
    #     6.6 - Se o valor na coluna [AUX_TIPO_ESTACAO_GEOPLAN] for igual a A DEFINIR ou NOVO_TERRENO ou COMERCIAL e [AUX_STATUS_SISLIC] for igual a NÃO TEM LICENÇA ou PENDENTE INSTALAÇÃO, mantém os status atuais e nos campos vazios mudar para "SEM CONEXÃO DE REDE"
    classificar_ERs()

    # 7.1 - Se o valor na coluna [STATUS CONSOLIDADO] for igual a OPERACIONAL e [AUX_STATUS_INFRAGEST] for igual a CONECTADO ou PENDENTE ATIVAÇÃO, então [STATUS_FATURAMENTO] é igual a 1
    #     7.2 - Se o valor na coluna [STATUS CONSOLIDADO] for igual a OPERACIONAL e [AUX_STATUS_INFRAGEST] for igual a INFRA NÃO CAPACITADA, então [STATUS_FATURAMENTO] é igual a 0
    #     7.3 - Para qualquer caso que não bata com as duas condições anteriores, [STATUS_FATURAMENTO] é igual a 2
    definir_faturamento()

    pass